In [1]:
!pip install ijson==3.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import ijson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from glob import glob

# unless pre preprocessing changes, no need to run above

In [4]:
imu_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

insole_sensor_locations = ['Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes']

participant_num = [1, 2, 3, 4, 5, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25]

In [5]:
import os
import pandas as pd
import logging
import glob

# Set up logging
logging.basicConfig(level=logging.INFO)

def merge_csv_files_in_folder(folder_path, insole_sensor_locations):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

    filtered_files = [file for file in csv_files if any(sensor_location in file for sensor_location in insole_sensor_locations)]

    if filtered_files:
        try:
            df_list = [pd.read_csv(file, low_memory=False) for file in filtered_files]
            merged_df = pd.concat(df_list, ignore_index=True)
            return merged_df
        except FileNotFoundError as e:
            logging.error(f"Error: File not found: {e}")
        except pd.errors.ParserError as e:
            logging.error(f"Error: Parsing error: {e}")
    else:
        logging.warning(f"No relevant CSV files found in {folder_path}")
        return None

In [6]:
# Load and process dataframes
folder_path = "/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/processed data"

insole_df = merge_csv_files_in_folder(folder_path, insole_sensor_locations)


In [7]:
insole_df.sort_values(by=['participant_id', 'time', 'sensor_location'])

,time,participant_id,sensor_location,task,surface,insoles_RightFoot_is_step,insoles_LeftFoot_is_step,insoles_RightFoot_is_lifted,insoles_LeftFoot_is_lifted,Left,Right,Right__raw,Left__raw,Left__norm,Right__norm
4062918,0,1,Arch,A,walk,True,True,False,False,406.0,465.0,3839.406648,4093.000000,0.001299,0.160105
4104398,0,1,Hallux,A,walk,True,True,False,False,3.0,0.0,2782.670582,4094.362659,0.000223,0.968475
4187358,0,1,Heel_L,A,walk,True,True,False,False,286.0,393.0,4094.362659,3611.384155,0.457522,0.000275
4145878,0,1,Heel_R,A,walk,True,True,False,False,430.0,437.0,4090.274681,3485.834793,0.372672,0.002728
4270318,0,1,Met1,A,walk,True,True,False,False,1.0,1.0,3424.637341,4091.318670,0.001484,0.987995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548117,771750,25,Heel_R,A,walk,False,False,False,False,422.0,8.0,4081.568781,3108.550972,0.868584,0.004348
3594423,771750,25,Met1,A,walk,False,False,False,False,287.0,6.0,3233.706342,4091.146829,0.001864,0.699607
3640729,771750,25,Met3,A,walk,False,False,False,False,388.0,4.0,2835.586590,4092.926586,0.000800,0.741234
3687035,771750,25,Met5,A,walk,False,False,False,False,415.0,9.0,3694.779757,4094.357805,0.000682,0.438408


In [8]:
# Function to pair heel strikes with the next toe-off event
def pair_heel_strikes_toe_offs(heel_strikes, toe_offs):
    pairs = []
    toe_idx = 0
    # Loop over each heel strike
    for heel in heel_strikes:
        # Find the next toe-off that occurs after the heel strike
        while toe_idx < len(toe_offs) and toe_offs[toe_idx] < heel:
            toe_idx += 1
        if toe_idx < len(toe_offs):
            # Pair found
            pairs.append((heel, toe_offs[toe_idx]))
            toe_idx += 1  # Move to the next toe-off for the next pair
    return pairs


In [9]:
# Initialize or clear gait cycle columns for each new run
insole_df['right_gait_cycle'] = np.nan
insole_df['left_gait_cycle'] = np.nan

for p in insole_df['participant_id'].unique():

    current_foot_sensors_df = insole_df.loc[insole_df['participant_id'] == p]

    # Define heel strikes and toe-offs for the right foot
    right_foot_heel_strikes = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_RightFoot_is_step'] == True) &
        (current_foot_sensors_df['insoles_RightFoot_is_lifted'] == False)]
    right_foot_toe_off = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_RightFoot_is_step'] == False) &
        (current_foot_sensors_df['insoles_RightFoot_is_lifted'] == True)]

    # Define heel strikes and toe-offs for the left foot
    left_foot_heel_strikes = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_LeftFoot_is_step'] == True) &
        (current_foot_sensors_df['insoles_LeftFoot_is_lifted'] == False)]
    left_foot_toe_off = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_LeftFoot_is_step'] == False) &
        (current_foot_sensors_df['insoles_LeftFoot_is_lifted'] == True)]

    # Convert heel strikes and toe-offs to lists
    heel_strike_R = list(right_foot_heel_strikes['time'])
    toe_off_R = list(right_foot_toe_off['time'])

    heel_strike_L = list(left_foot_heel_strikes['time'])
    toe_off_L = list(left_foot_toe_off['time'])

    # Generate pairs of steps for right and left foot
    right_foot_steps = pair_heel_strikes_toe_offs(heel_strike_R, toe_off_R)
    left_foot_steps = pair_heel_strikes_toe_offs(heel_strike_L, toe_off_L)

    # Enumerate and assign step cycles to the DataFrame
    for i, (start, end) in enumerate(right_foot_steps):
        insole_df.loc[
            (insole_df['participant_id'] == p) &
            (insole_df['time'] >= start) &
            (insole_df['time'] <= end), 'right_step_count'] = i + 1

    for i, (start, end) in enumerate(left_foot_steps):
        insole_df.loc[
            (insole_df['participant_id'] == p) &
            (insole_df['time'] >= start) &
            (insole_df['time'] <= end), 'left_step_count'] = i + 1

    print(f"Processed participant {p}")


Processed participant 4
Processed participant 17
Processed participant 10
Processed participant 2
Processed participant 18
Processed participant 8
Processed participant 22
Processed participant 23
Processed participant 7
Processed participant 14
Processed participant 12
Processed participant 24
Processed participant 25
Processed participant 15
Processed participant 1
Processed participant 16
Processed participant 13
Processed participant 5
Processed participant 3
Processed participant 19


In [10]:
insole_df['right_gait_cycle'] = insole_df['right_step_count'].fillna(method='ffill')
insole_df['left_gait_cycle'] = insole_df['left_step_count'].fillna(method='ffill')


<ipython-input-10-7787957a4271>:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  insole_df['right_gait_cycle'] = insole_df['right_step_count'].fillna(method='ffill')
<ipython-input-10-7787957a4271>:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  insole_df['left_gait_cycle'] = insole_df['left_step_count'].fillna(method='ffill')


In [11]:
insole_df['participant_id'].unique()

array([ 4, 17, 10,  2, 18,  8, 22, 23,  7, 14, 12, 24, 25, 15,  1, 16, 13,
        5,  3, 19])

In [12]:
insole_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/insole_sensor_df.csv', index=False)